# Extração do Dataset: Bolsa Família

**Fonte:** `https://dados.gov.br/dados/conjuntos-dados/bolsa-familia---beneficios-basicos-e-variaveis`

**Objetivo:** Obter a quantidade de famílias beneficiárias agregadas por Código IBGE do município

In [1]:
import pandas as pd

# Insira o nome exato do arquivo CSV que você baixou do portal dados.gov.br:
ARQUIVO_CSV = "data/beneficios_bolsa_familia.csv"

print("Iniciando leitura do CSV Local...")

# Bases do governo costumam usar sep=';' e encoding latin-1
try:
    df_bolsa = pd.read_csv(ARQUIVO_CSV, sep=',', encoding='latin-1', low_memory=False)
    print("Arquivo lido com sucesso usando latin-1 e separador ';'")
except FileNotFoundError:
    print(f"\nERRO: O arquivo '{ARQUIVO_CSV}' não foi encontrado na pasta atual.")
    print("Por favor, acesse o dados.gov.br, baixe o CSV e o coloque nesta mesma pasta para continuarmos.")
    # Cria um dataframe vazio para não quebrar o resto do notebook no 'Run All'
    df_bolsa = pd.DataFrame()
except Exception as e:
    print("Tentando leitura com codificação alternativa (utf-8 e virgula)...")
    df_bolsa = pd.read_csv(ARQUIVO_CSV, sep=',', encoding='utf-8', low_memory=False)

if not df_bolsa.empty:
    print(f"\nDimensão original do Dataset Histórico: {df_bolsa.shape}")
    # Padronizar nomes das colunas (tudo minúsculo e sem espaços)
    df_bolsa.columns = df_bolsa.columns.str.lower().str.strip()
    display(df_bolsa.head())
else:
    print("\nA execução foi interrompida pois o arquivo não foi carregado.")

Iniciando leitura do CSV Local...
Arquivo lido com sucesso usando latin-1 e separador ';'

Dimensão original do Dataset Histórico: (66852, 9)


,ibge,siglauf,anomes,qtd_ben_bas,qtd_ben_var,qtd_ben_bvj,qtd_ben_bvn,qtd_ben_bvg,qtd_ben_bsp
0,110001,RO,202101,1339.0,2301.0,251.0,20.0,43.0,247.0
1,110001,RO,202102,1098.0,2321.0,248.0,30.0,47.0,262.0
2,110001,RO,202103,1113.0,2329.0,254.0,35.0,51.0,263.0
3,110001,RO,202104,1123.0,2328.0,261.0,41.0,51.0,255.0
4,110001,RO,202105,1123.0,2328.0,261.0,41.0,51.0,255.0


## 3. Filtrar Exclusivamente para o Ano de 2021
O dataset histórico contém muitos anos. Vamos isolar 2021 para montar o painel unificado requisitado no estudo de caso.

In [2]:
if not df_bolsa.empty:
    # Identificar a coluna de ano/referência.
    coluna_data = None
    for possivel_coluna in ['ano_referencia', 'mes_referencia', 'data_referencia', 'referencia', 'ano']:
        if possivel_coluna in df_bolsa.columns:
            coluna_data = possivel_coluna
            break

    if coluna_data:
        # Garantir que é string para filtrar ano
        df_bolsa[coluna_data] = df_bolsa[coluna_data].astype(str)
        df_2021 = df_bolsa[df_bolsa[coluna_data].str.startswith('2021')].copy()
        print(f"Encontrados {df_2021.shape[0]} registros para o ano de 2021 na coluna '{coluna_data}'.")
    else:
        print("Aviso: Não achou coluna temporal clara. Exibindo colunas para inspeção:", df_bolsa.columns)
        df_2021 = df_bolsa.copy()
else:
    df_2021 = pd.DataFrame()

Aviso: Não achou coluna temporal clara. Exibindo colunas para inspeção: Index(['ibge', 'siglauf', 'anomes', 'qtd_ben_bas', 'qtd_ben_var',
       'qtd_ben_bvj', 'qtd_ben_bvn', 'qtd_ben_bvg', 'qtd_ben_bsp'],
      dtype='object')


## 4. Agrupamento (Target)
Para criar um modelo preditivo geolocalizado, precisaremos de **1 linha = 1 município**. 
Agrupamos os repasses mensais de 2021 tirando uma média (famílias beneficiárias médias daquele município no ano).

In [3]:
if not df_2021.empty:
    # Identificar a coluna com o Código IBGE do município
    col_ibge = [col for col in df_2021.columns if 'ibge' in col.lower()]
    col_ibge = col_ibge[0] if col_ibge else None

    if col_ibge:
        print(f"Agrupando pela coluna geográfica de IBGE: '{col_ibge}'")
        
        # Selecionar variáveis numéricas que remetem a benefícios ou famílias
        cols_numericas = df_2021.select_dtypes(include=['float64', 'int64']).columns
        
        # Fazer a média das métricas durante todos os meses de 2021 por município
        df_target = df_2021.groupby(col_ibge)[cols_numericas].mean().reset_index()
        
        # Renomear para padronizar com o SIDRA
        df_target = df_target.rename(columns={col_ibge: 'cod_municipio'})
        
        # Se existirem as colunas básicas do Governo 
        cols_beneficios = [c for c in df_target.columns if 'qtd' in c or 'ben' in c or 'familias' in c]
        if cols_beneficios:
            df_target['TARGET_qtd_familias_bolsa_familia_media_2021'] = df_target[cols_beneficios].sum(axis=1)
            
        print(f"\nDimensão Final da Tabela Target: {df_target.shape}")
        display(df_target.head(10))
        
    else:
        print("Erro: Não foi possível identificar uma coluna com o Código IBGE no dataset baixado.")
        display(df_2021.head())
else:
    print("Processamento cancelado devido à ausência do dataset.")


Agrupando pela coluna geográfica de IBGE: 'ibge'


ValueError: cannot insert ibge, already exists

## 5. Próximos Passos (Para Atuação do Especialista)
Agora você tem disponível neste dataframe a coluna de ligação `cod_municipio` e a métrica alvo (`TARGET_qtd_familias_bolsa_familia_media_2021`).

Basta utilizar o `pandas.merge()` entre esta base e o `df_final` gerado no script do SIDRA para unificar suas *Features Socioeconômicas* com o seu *Target de Pobreza* e treinar seu algoritmo supervisionado.